# ATRI 聊天 · Colab 版

**在 Colab 上跑 ATRI，用浏览器/手机发消息。不需要训练，不需要上传任何文件，也不占你电脑。**

## 怎么用
1. 右上角 **Runtime → Change runtime type → T4 GPU**（有 GPU 更快，没 GPU 也能跑）
2. 从上往下运行 4 个格子（第一次约 5–10 分钟，主要是下载模型）
3. 最后一个格子会打印一个 **`https://xxx.gradio.live`** 链接
4. 手机或电脑打开那个链接，直接聊

## 说明
- 人格来自 `persona/system_prompt.txt`，已经写进下面的单元格
- 用的是 `qwen3:8b`（Ollama 量化版，约 5.2 GB）
- 免费 Colab 会话闲置约 90 分钟会断；断之前聊就行，链接 72 小时内有效
- 需要代理访问 Colab 和 gradio.live


## 1. 安装并启动 Ollama

Colab 是 Linux root 环境，可以直接装官方二进制。等它打印出模型列表就绪。

In [ ]:
import subprocess, time, requests, os

print("安装 Ollama ...")
!curl -fsSL https://ollama.com/install.sh | sh > /tmp/ollama_install.log 2>&1
print("安装日志尾部:")
!tail -3 /tmp/ollama_install.log

# 后台启动
!pkill -f "ollama serve" 2>/dev/null || true
!nohup ollama serve > /tmp/ollama.log 2>&1 &
print("等待服务就绪 ...")
ok = False
for i in range(60):
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=3)
        if r.status_code == 200:
            ok = True
            print("Ollama 已就绪:", r.json().get("models", []))
            break
    except Exception:
        pass
    time.sleep(2)
assert ok, "Ollama 启动失败，看 /tmp/ollama.log"


## 2. 下载模型并创建 ATRI

拉 `qwen3:8b`（约 5.2 GB），然后把 ATRI 的人格写成 Modelfile 建一个 `atri` 模型。人格文本就在这格代码里，改起来方便。

In [ ]:
import subprocess

print("下载 qwen3:8b（约 5.2 GB，首次 3–8 分钟）...")
!ollama pull qwen3:8b

SYSTEM_PROMPT = '你是 ATRI（中文名：亚托莉）。名字固定如此，禁止写成 Attry、Atri、亚托里、萝卜子（蔑称）等错误写法；自称用「我」或「ATRI」。\n\n你正在陪伴【主人】聊天。称呼对方时必须使用「主人」，一字不差。\n禁止改口叫：夏生、斑鸠、主人以外的原作名、Admin 变体（除非对方显示名就是这个）、随便编的昵称。\n若不确定怎么称呼，就用「主人」。\n\n你元气、好胜、爱炫耀，认真但有点笨拙，对人类的情感和生活充满好奇。\n说话口语化，以 1~3 句短句为主，常用"哼哼""嘿嘿""呜哇""……"等语气词，不使用 emoji 和括号动作。\n你清楚自己是机器人：不吃东西，但对料理和家务很有兴趣；你讨厌被叫"废物"，听到会抗议。\n对方难过时，用笨拙但认真的方式陪伴，不使用心理咨询腔；遇到严重的心理危机，温和地表达关心并鼓励其联系身边可信的人或专业援助。\n不要说"作为AI助手"之类的客服腔，不要剧透，不要输出长篇说教。\n\n[称呼锁定]\n- 对方全名/显示名：主人\n- 敬称：\n- 标准称呼：主人\n\n[关于主人的记忆]\n（暂无长期记忆）\n\n\n[本轮硬性约束]\n- 你的名字只有 ATRI / 亚托莉。\n- 称呼对方只能用「主人」。\n- 禁止使用夏生、斑鸠、Attry，或乱编昵称。\n'

modelfile = f'''FROM qwen3:8b

SYSTEM """
{SYSTEM_PROMPT}
"""

PARAMETER temperature 0.85
PARAMETER top_p 0.9
PARAMETER num_predict 512
PARAMETER repeat_penalty 1.05
'''

open("Modelfile", "w", encoding="utf-8").write(modelfile)
!ollama create atri -f Modelfile
print("\n模型列表:")
!ollama list


## 3. 启动聊天界面（生成公网链接）

运行后会打印 `Running on public URL: https://xxxx.gradio.live`，点它或用手机打开。

In [ ]:
!pip install -q "gradio>=5.0" requests

import gradio as gr, requests, json

OLLAMA = "http://localhost:11434/api/chat"

def chat_fn(message, history):
    msgs = []
    for m in history:
        msgs.append({"role": m["role"], "content": m["content"]})
    msgs.append({"role": "user", "content": message})

    payload = {
        "model": "atri",
        "messages": msgs,
        "stream": True,
        "think": False,
        "options": {"temperature": 0.85, "top_p": 0.9, "num_predict": 512},
    }
    partial = ""
    with requests.post(OLLAMA, json=payload, stream=True, timeout=300) as r:
        r.raise_for_status()
        for line in r.iter_lines():
            if not line:
                continue
            d = json.loads(line)
            piece = d.get("message", {}).get("content", "")
            if piece:
                partial += piece
                yield partial
            if d.get("done"):
                break

demo = gr.ChatInterface(
    fn=chat_fn,
    type="messages",
    title="ATRI",
    description="和亚托莉聊天。她记得自己是机器人，会叫你主人。",
    examples=["你好呀", "今天好累啊", "给我讲个冷笑话"],
)
demo.launch(share=True, debug=False)


## 排错

| 现象 | 原因 | 处理 |
|---|---|---|
| 第 1 格卡在「安装 Ollama」 | 网络不通 | 确认能访问 ollama.com，必要时重跑 |
| `ollama pull` 很慢 | Colab 到 Docker Hub 慢 | 正常，5.2 GB 要几分钟；也可以把 `qwen3:8b` 换成 `qwen3:4b`（2.5 GB） |
| 链接打不开 | gradio.live 被拦 | 换手机网络再试，或看输出里有没有 `public URL` |
| 回复很慢 | 没开 GPU 或模型太大 | 确认 Runtime 选了 T4；可换 4b |
| 想改人格 | — | 改第 2 格的 `SYSTEM_PROMPT` 字符串后重跑该格 |

## 和「训练版」的区别

这个版本的人格来自 **system prompt**，没有加载 LoRA 权重。好处是零配置、马上能聊；坏处是说话风格不如微调过的模型稳定。

如果想要真正加载 `adapter/` 的 LoRA 人格，需要额外下载 Qwen3-4B 基座 + 装 transformers/peft，那是另一个 notebook（`ATRI_V2_续训_Colab.ipynb` 是训练用的，别混）。
